# Challenge 2: Predicting Emergency Call Response Times with BigQuery ML

In [4]:
from google.cloud import bigquery

PROJECT_ID = "qwiklabs-gcp-00-f469c11ef7ce"
DATASET = "emergency_response"
SOURCE_URI = "gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv"

client = bigquery.Client(project=PROJECT_ID, location="US")

connected


In [5]:
RAW_TABLE = f"{PROJECT_ID}.{DATASET}.calls_raw"

client.query(f"CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET}` OPTIONS(location='US')").result()

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)
client.load_table_from_uri(SOURCE_URI, RAW_TABLE, job_config=job_config).result()

table = client.get_table(RAW_TABLE)
print(f"{table.num_rows:,} rows, {len(table.schema)} columns")

50,000 rows, 11 columns


In [6]:
for f in table.schema:
    print(f"{f.name:35s} {f.field_type}")
print("-" * 50)
client.query(f"SELECT * FROM `{RAW_TABLE}` LIMIT 5").to_dataframe()

call_id                             INTEGER
call_timestamp                      TIMESTAMP
call_type                           STRING
location                            STRING
weather_condition                   STRING
day_of_week                         STRING
time_of_day                         INTEGER
traffic_level                       STRING
distance_to_station                 FLOAT
units_available                     INTEGER
response_time                       FLOAT
--------------------------------------------------


,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37


In [7]:
MODEL = f"{PROJECT_ID}.{DATASET}.response_time_model"

create_model_sql = f"""
CREATE OR REPLACE MODEL `{MODEL}`
OPTIONS(
  model_type = 'linear_reg',
  input_label_cols = ['response_time'],
  data_split_method = 'AUTO_SPLIT',
  enable_global_explain = TRUE
) AS
SELECT
  call_type,
  location,
  weather_condition,
  day_of_week,
  time_of_day,
  traffic_level,
  distance_to_station,
  units_available,
  response_time
FROM `{RAW_TABLE}`
"""

client.query(create_model_sql).result()
print(f"Model trained -> {MODEL}")

Model trained -> qwiklabs-gcp-00-f469c11ef7ce.emergency_response.response_time_model


In [8]:
client.query(f"SELECT * FROM ML.EVALUATE(MODEL `{MODEL}`)").to_dataframe()


,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.761934,4.827846,0.015117,1.501986,0.831417,0.83146


In [9]:
client.query(f"SELECT * FROM ML.GLOBAL_EXPLAIN(MODEL `{MODEL}`)").to_dataframe()

,feature,attribution
0,location,1185.022651
1,call_type,447.977976
2,day_of_week,240.382238
3,weather_condition,169.816521
4,distance_to_station,3.403623
5,traffic_level,2.320936
6,units_available,0.526156
7,time_of_day,0.001006


In [10]:
predict_sql = f"""
SELECT *
FROM ML.PREDICT(MODEL `{MODEL}`,
  (SELECT
     'Fire' AS call_type,
     'Riverside' AS location,
     'Rainy' AS weather_condition,
     'Monday' AS day_of_week,
     8 AS time_of_day,
     'High' AS traffic_level,
     20.5 AS distance_to_station,
     2 AS units_available)
)
"""
client.query(predict_sql).to_dataframe()

,predicted_response_time,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available
0,22.104897,Fire,Riverside,Rainy,Monday,8,High,20.5,2
